In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
print("Imported successfully")

In [ ]:
# load dataset
df = pd.read_csv(r"D:\Project_programming\case_study_mc\Data\processesed\hse_data.csv")
print("Loaded successfully")

In [ ]:
# global parameters
sns.set_style('darkgrid')
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor':'#444444',
    'axes.labelcolor':'#333333',
    'xtick.color':'#333333',
    'ytick.color':'#333333',
})

In [ ]:
# sample size
df.groupby('year').size()

In [ ]:
# determine male and female composition
# 1.0 - Male
# 2.0 - Female

sex_summary = (
  pd.crosstab(
      df['year'],
      df['sex'],
      normalize = 'index'
  )*100
)
sex_summary.round(2)

In [ ]:
# smoking status between the two datasets
# 1 "Never smoked cigarettes at all" 
# 2 "Used to smoke cigarettes occasionally" 
# 3 "Used to smoke cigarettes regularly" 
# 4 "Current cigarette smoker". 
# negative smoker - (data anomalies)
absolute_smoking_status = (
  pd.crosstab(
    df['year'],
    df['smoking_status'],
  )
)
absolute_smoking_status

In [ ]:
# create a new column for current smokers
# 1.0 - current smokers
# 0.0 - used to smoke either regularly or occassionally
df.loc[
  df['smoking_status'] == 4.0,
  'current_smoker'
] = 1.0
df.loc[
  (
    (df['smoking_status'] == 2.0)
    |
    (df['smoking_status'] == 3.0)
  ),
  'current_smoker'
] = 0.0
df['current_smoker'] = pd.to_numeric(df['current_smoker'],errors='coerce')
df['current_smoker'].value_counts()

In [ ]:
# calculate smoking prevalence
smoking_summary = (
    df.groupby("year")
       .agg(
           sample_size=("current_smoker", "count"),
           smokers=("current_smoker", "sum"),
           prevalence=("current_smoker", "mean")
       )
)

smoking_summary["prevalence"] *= 100

display(smoking_summary)

In [ ]:
p99 = smoking_summary.loc[1999, "prevalence"]
p19 = smoking_summary.loc[2019, "prevalence"]

absolute_change = p19 - p99

relative_change = (
    (p19 - p99) / p99
) * 100

print("1999 prevalence:", p99)
print("2019 prevalence:", p19)
print("Absolute change (percentage points):", absolute_change)
print("Relative change (%):", relative_change)

In [ ]:
# Find which gender actively smokes
gender_trend = (
    df.groupby(["year", "sex"])
       .agg(
            sample_size=("current_smoker", "count"),
            smokers=("current_smoker", "sum"),
            gender_prevalence=("current_smoker", "mean")
       )
       .unstack()
       .reset_index()
)
gender_trend['gender_prevalence']*=100
display(gender_trend)

In [ ]:
# Age distribution where they started to regularly smoke
# plot a histogram
fig, ax = plt.subplots(2,1 ,figsize = (12,7))

# histogram for 1999
sns.histplot(
  data = df.loc[df['year'] == 1999],
  x = df['age_started_regular_smoking'],
  bins= 100,
  edgecolor = 'black',
  ax = ax[0]
)
# histogram for 2019
sns.histplot(
  data = df.loc[df['year'] ==2019],
  x = df['age_started_regular_smoking'],
  bins= 100,
  edgecolor = 'black',
  ax = ax[1]
)
ax[0].set_title('Age distribution of regular smoking initiation (1999)',
                fontweight='bold', fontsize=14)

ax[1].set_title('Age distribution of regular smoking initiation (2019)',
                fontweight='bold', fontsize=14)

for a in ax:
    a.set_ylabel('Frequency',
                 fontsize=12, fontweight='bold')
    a.set_xlabel('Age', fontsize = 12, fontweight = 'bold')

fig.subplots_adjust(hspace=0.30)
# Check frequency distribution
# print(df['age_started_regular_smoking'].value_counts())
print(df['age_started_regular_smoking'].value_counts())

In [ ]:
#  "No answer/refused" -8 "Don't know" -1 "Item not applicable" 
#  0 "Has never smoked"  
#  1 "Ever smoked but not currently smoking a cigar or pipe" 
#  2 "Ever smoked but questions about cigar or pipe not applicable" 
#  3 "Currently smokes a cigar or pipe". 
# Absolute current smokers of cigars summary
cigar_summary = (
  pd.crosstab(
      df['year'],
      df['current_cigar_smoker']
  )
)
cigar_summary

In [ ]:
# Absolute current smokers of pipe summary
pipe_summary = (
    pd.crosstab(
        df['year'],
        df['current_pipe_smoker'],
    )
)
pipe_summary

In [ ]:
# save as csv file
df.to_csv(r"D:\Project_programming\case_study_mc\Data\processesed\hse2_data.csv")
print("Saved successfully!")